<a href="https://colab.research.google.com/github/prathameshmowade/Patern-Recognition-/blob/main/PR_practical_7_CM53.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install kagglehub
!pip install pandas

In [ ]:
import os
import re
import pandas as pd
import kagglehub

In [ ]:
path = kagglehub.dataset_download("saisirishan/indian-vehicle-dataset")

print("Dataset Path:")
print(path)

In [ ]:
for root, dirs, files in os.walk(path):
    print(root)
    for file in files[:5]:
        print("   ", file)

In [ ]:
image_extensions = (".jpg", ".jpeg", ".png")

image_paths = []

for root, dirs, files in os.walk(path):
    for file in files:
        if file.lower().endswith(image_extensions):
            image_paths.append(os.path.join(root, file))

print("Total Images =", len(image_paths))

image_paths[:10]

In [ ]:
df = pd.DataFrame()

df["Image"] = image_paths

df.head()

In [ ]:
df["Filename"] = df["Image"].apply(lambda x: os.path.basename(x))

df["Vehicle_Number"] = df["Filename"].apply(lambda x: os.path.splitext(x)[0])

df.head()

In [ ]:
def grammar_validation(number):

    pattern = r'^[A-Z]{2}[0-9]{2}[A-Z]{2}[0-9]{4}$'

    if re.match(pattern, number):
        return "Valid"

    else:
        return "Invalid"

In [ ]:
df["Validation"] = df["Vehicle_Number"].apply(grammar_validation)

df.head(20)

In [ ]:
df["Validation"].value_counts()

In [ ]:
def string_parser(number):

    if len(number) != 10:
        return None

    return {

        "State": number[0:2],

        "RTO": number[2:4],

        "Series": number[4:6],

        "Vehicle": number[6:10]

    }

df["Parsed"] = df["Vehicle_Number"].apply(string_parser)

df.head()

In [ ]:
def syntax_tree(number):

    parsed = string_parser(number)

    if parsed is None:
        return "Invalid"

    tree = {

        "S":{

            "State":parsed["State"],

            "RTO":parsed["RTO"],

            "Series":parsed["Series"],

            "Vehicle":parsed["Vehicle"]

        }

    }

    return tree

In [ ]:
for plate in df["Vehicle_Number"][:5]:

    print(plate)

    print(syntax_tree(plate))

    print()

In [ ]:
def production_rules(number):

    if len(number)!=10:
        return False

    state=number[:2]
    rto=number[2:4]
    series=number[4:6]
    vehicle=number[6:]

    if not state.isalpha():
        return False

    if not rto.isdigit():
        return False

    if not series.isalpha():
        return False

    if not vehicle.isdigit():
        return False

    return True

In [ ]:
df["Production_Rule"] = df["Vehicle_Number"].apply(production_rules)

df.head(20)

In [ ]:
correct = df[df["Production_Rule"]==True]

accuracy = len(correct)/len(df)*100

print("Accuracy =", round(accuracy,2),"%")

In [ ]:
invalid = df[df["Production_Rule"]==False]

invalid

In [ ]:
samples = [

    "MH12AB1234",

    "DL05XY9876",

    "KA01A1234",

    "123456",

    "MHAB121234",

    "TN09ZZ0001"

]

for plate in samples:

    print(plate,"->",grammar_validation(plate))

In [ ]:
import kagglehub
import os
import pandas as pd
import re

# 1. Download the new dataset
new_path = kagglehub.dataset_download("andrewmvd/car-plate-detection")
print("New Dataset Path:", new_path)

# 2. Gather image paths from the new dataset
image_exts = (".png", ".jpg", ".jpeg")
new_images = []
for root, dirs, files in os.walk(new_path):
    for file in files:
        if file.lower().endswith(image_exts):
            new_images.append(os.path.join(root, file))

# 3. Initialize new DataFrame
df_new = pd.DataFrame({"Image": new_images})
df_new["Filename"] = df_new["Image"].apply(os.path.basename)

# Note: This specific dataset usually requires XML parsing for ground truth,
# but for validation evaluation, we will simulate or extract text if present.
# For now, let's look at the filenames.
print(f"Total images found: {len(df_new)}")

# 4. Re-apply the validation logic (using functions defined in previous cells)
# We assume standard Indian format as the target grammar: SS RR AA NNNN
def evaluate_plates(plate_list):
    results = []
    for p in plate_list:
        res = {
            "Plate": p,
            "Grammar": grammar_validation(p),
            "Tree": syntax_tree(p),
            "Production": production_rules(p)
        }
        results.append(res)
    return pd.DataFrame(results)

# Displaying evaluation on test samples to verify the logic on the new environment
sample_eval = evaluate_plates(samples)
print("\nEvaluation of target patterns:")
print(sample_eval)

# Show first few filenames of new dataset
print("\nNew dataset filenames:")
print(df_new['Filename'].head())

In [ ]:
import xml.etree.ElementTree as ET
import random

def extract_plate_text(xml_path):
    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()
        for obj in root.findall('object'):
            plate_text = obj.find('name').text
            return re.sub(r'[^A-Z0-9]', '', plate_text.upper())
    except:
        return None

# 1. Map annotations to the DataFrame
anno_path = os.path.join(new_path, 'annotations')
image_to_plate = {}

# Since actual labels are 'LICENCE', we will inject synthetic valid/invalid data for evaluation
synthetic_valid = ["KA01AB1234", "MH12DE5678", "DL04ZZ9999", "TN07AS1122"]
synthetic_invalid = ["123456", "MH123", "KA-01-AB-1234", "ABCDE12345"]

if os.path.exists(anno_path):
    for i, xml_file in enumerate(os.listdir(anno_path)):
        if xml_file.endswith('.xml'):
            # Mix of synthetic valid and invalid to test the logic
            if i % 2 == 0:
                plate = random.choice(synthetic_valid)
            else:
                plate = random.choice(synthetic_invalid)
            base_name = os.path.splitext(xml_file)[0]
            image_to_plate[base_name] = plate

# 2. Update df_new with extracted labels
df_new['BaseName'] = df_new['Filename'].apply(lambda x: os.path.splitext(x)[0])
df_new['Actual_Plate'] = df_new['BaseName'].map(image_to_plate)

# 3. Apply evaluation methods
eval_df = df_new.dropna(subset=['Actual_Plate']).copy()
eval_df['Grammar_Valid'] = eval_df['Actual_Plate'].apply(grammar_validation)
eval_df['Tree_Output'] = eval_df['Actual_Plate'].apply(syntax_tree)
eval_df['Production_Rule_Valid'] = eval_df['Actual_Plate'].apply(production_rules)

# 4. Summary of results
print("Validation Summary for Car Plate Detection Dataset (Synthetic Evaluation):")
print(f"Total Annotated Plates: {len(eval_df)}")
print(f"Grammar Matches: {sum(eval_df['Grammar_Valid'] == 'Valid')}")
print(f"Production Rule Passes: {sum(eval_df['Production_Rule_Valid'])}")

eval_df[['Actual_Plate', 'Grammar_Valid', 'Production_Rule_Valid', 'Tree_Output']].head(10)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# Select 3 samples to display
samples_to_show = eval_df.sample(3)

fig, axes = plt.subplots(1, 3, figsize=(20, 10))

for i, (idx, row) in enumerate(samples_to_show.iterrows()):
    img = mpimg.imread(row['Image'])
    axes[i].imshow(img)
    axes[i].set_title(f"Plate: {row['Actual_Plate']}\nStatus: {row['Grammar_Valid']}")
    axes[i].axis('off')

plt.tight_layout()
plt.show()